---
title: Quality Control Checks
date: 2026-03-06
---

This notebook demonstrates the `xopr.qc` module, which provides standardized quality control checks for radar datasets. Each check produces a per-trace boolean mask that can be used to filter or visualize data quality.

We'll load flight lines over the David Glacier region and apply an ice thickness threshold check, then shade the failing regions on a radargram.

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

import xopr
from xopr.qc import run_qc, ice_thickness_threshold

In [ ]:
opr = xopr.OPRConnection(cache_dir="radar_cache")

## Find flight lines over the David Glacier region

We use `xopr.geometry.get_antarctic_regions` to get the David Glacier boundary from the MEaSUREs dataset, then query for radar frames that intersect it.

In [ ]:
region = xopr.geometry.get_antarctic_regions(name="David", merge_regions=True)
stac_items = opr.query_frames(geometry=region, max_items=20)
print(f"Found {len(stac_items)} frames in the David region")
stac_items.head()

## Load a flight line

We'll load all frames from the first segment in our results, merge them into one dataset, and resample to uniform 2-second spacing.

In [ ]:
# Identify the first segment in the results
first_item = stac_items.iloc[0]
collection = first_item['collection']
opr_date = first_item['properties']['opr:date']
opr_segment = first_item['properties']['opr:segment']
print(f"Loading segment {opr_date}_{opr_segment:02d} from {collection}")

# Filter to all frames from this segment
segment_items = stac_items[
    stac_items['properties'].apply(
        lambda p: p['opr:date'] == opr_date and p['opr:segment'] == opr_segment
    ) & (stac_items['collection'] == collection)
]
print(f"Found {len(segment_items)} frames in this segment")

frames = opr.load_frames(segment_items)
flight_line = xopr.merge_frames(frames)
flight_line = flight_line.resample(slow_time='2s').mean()
flight_line.xopr

## Run QC checks

`run_qc()` automatically loads layer picks (surface and bottom) when they are not already in the dataset. Just pass the `opr` connection and it handles the rest.

The `ice_thickness_threshold` check computes ice thickness from the `Surface` and `Bottom` travel-time picks and flags traces where thickness falls below a minimum (default: 500 m). Traces with NaN picks are also flagged as failing.

In [ ]:
qc_ds = run_qc(
    flight_line,
    opr=opr,
    check_params={"ice_thickness_threshold": {"min_thickness_m": 500}},
)

n_pass = qc_ds["qc"].sum().item()
n_total = qc_ds.sizes["slow_time"]
print(f"QC result: {n_pass}/{n_total} traces passed ({100*n_pass/n_total:.1f}%)")

## Visualize the radargram with QC overlay

We plot the radargram in grayscale and shade traces that fail QC with a semi-transparent red overlay.

In [ ]:
radargram = 10 * np.log10(np.abs(qc_ds["Data"]))

fig, ax = plt.subplots(figsize=(15, 5))
radargram.plot.imshow(x="slow_time", cmap="gray", ax=ax)
ax.invert_yaxis()

# Plot surface and bottom picks from the QC dataset
if "standard:surface" in qc_ds:
    qc_ds["standard:surface"].plot(ax=ax, x="slow_time", color="cyan", linewidth=0.5, label="Surface")
if "standard:bottom" in qc_ds:
    qc_ds["standard:bottom"].plot(ax=ax, x="slow_time", color="yellow", linewidth=0.5, label="Bottom")

# Shade contiguous runs of failing traces with red overlay
fail_mask = ~qc_ds["qc"].values
if fail_mask.any():
    slow_times = qc_ds.slow_time.values
    if len(slow_times) > 1:
        half_dt = (slow_times[1] - slow_times[0]) / 2
    else:
        half_dt = np.timedelta64(1, "s")
    # Find contiguous runs of failures
    diff = np.diff(fail_mask.astype(int))
    starts = np.where(np.concatenate(([fail_mask[0]], diff == 1)))[0]
    ends = np.where(np.concatenate((diff == -1, [fail_mask[-1]])))[0]
    for s, e in zip(starts, ends):
        ax.axvspan(slow_times[s] - half_dt, slow_times[e] + half_dt,
                   color="red", alpha=0.25)

ax.legend(loc="upper right")
ax.set_title(f"{collection} — {opr_date}_{opr_segment:02d}\nRed shading = failed QC (ice thickness < 500 m)")
plt.tight_layout()
plt.show()

## Using checks directly

You can also call `ensure_picks` and individual check functions directly instead of going through `run_qc()`. This is useful if you want to inspect the result of a single check or use non-default parameters:

In [ ]:
from xopr.qc import ensure_picks

# ensure_picks loads Surface/Bottom from layer data if they're not already present
ds_with_picks = ensure_picks(flight_line, opr=opr)

# Apply the check directly with a stricter threshold
strict_ds = ice_thickness_threshold(ds_with_picks, min_thickness_m=1000)

n_pass_strict = strict_ds["qc"].sum().item()
print(f"With 1000 m threshold: {n_pass_strict}/{n_total} traces passed ({100*n_pass_strict/n_total:.1f}%)")
print(f"With  500 m threshold: {n_pass}/{n_total} traces passed ({100*n_pass/n_total:.1f}%)")